![](https://fplogoimages.withfloats.com/actual/68009c3a43430aff8a30419d.png)
![](https://theciotimes.com/wp-content/uploads/2021/03/TELECOM1.jpg)

### 1. SQL Statements

In [0]:
%sql
create catalog if not exists telecom_catalog_assign;
create schema if not exists telecom_catalog_assign.landing_zone;
create volume if not exists telecom_catalog_assign.landing_zone.landing_vol;


In [0]:
dbutils.fs.mkdirs("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer")
dbutils.fs.mkdirs("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage")
dbutils.fs.mkdirs("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower")
dbutils.fs.ls("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol")

##### A. DBFS/FileStore is mainly for convenience and experimentation. It only has basic, workspace-level access control and lacks strong governance, auditing, and fine-grained security. Volumes, on the other hand, are part of Unity Catalog and provide centralized governance with access control and auditing, making them suitable for production and regulated data.
##### B. Volumes are part of unity catalog because production data requires centralized governance, strict access control, auditing which is ideal for regulated data, that is the reason production team prefers Volumes. 


### Data files to be copied

In [0]:
customer_csv = '''
101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID
'''

usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20
102\t120\t4000\t5
103\t540\t600\t52
104\t45\t200\t2
105\t0\t0\t0
'''



tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12
'''

### Copying the data in respective folders as csv,tsv file formats

In [0]:
dbutils.fs.put("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",customer_csv, overwrite=True)
dbutils.fs.put("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv",usage_tsv,overwrite=True)
dbutils.fs.put("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_region.csv",tower_logs_region1,overwrite=True)

### 2.File system operations

In [0]:
display(dbutils.fs.ls("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv"))
display(dbutils.fs.ls("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_region.csv"))
display(dbutils.fs.ls("dbfs:///Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv"))

### 3.Spark Directory Read Use Cases

In [0]:
df1_tower=(
    spark.read
    .options(pathGlobFilter="*.csv")
    .csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower")
)
display(df1_tower)

In [0]:
df2_tower=(
    spark.read
    .csv(path=["/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_region.csv"])
)
display(df2_tower)

In [0]:
df3_tower=(
    spark.read
    .option("recursiveFileLookup","True")
    .csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/")
)
display(df3_tower)

### 4. Schema Inference, Header and Separator

In [0]:
#options inside URI
df1_customer=(
  spark.read
  .csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv",header=False, inferSchema=False)#header and inferschema are False by default
)

#using options
display(df1_customer)
df2_customer=(
  spark.read
  .options(header=True, inferSchema=True)
  .csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv")
)
display(df2_customer)

#using option
df3_customer=(
  spark.read
  .option("header","False")
  .option("inferSchema","False")
  .csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv")
)
display(df3_customer)

#format 
df4_customer=(
  spark.read
  .format("csv")
  .option("header","True")
  .option("inferSchema","True")
  .csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv")
)
display(df4_customer)

In [0]:
#options inside URI
df1_usage=(
  spark.read
  .csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv",header=False, inferSchema=False)#header and inferschema are False by default
)

#using options
display(df1_usage)
df2_usage=(
  spark.read
  .options(header=True, inferSchema=True)
  .csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv")
)
display(df2_usage)

#using option
df3_usage=(
  spark.read
  .option("header","False")
  .option("inferSchema","False")
  .csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv")
)
display(df3_usage)

#format 
df4_usage=(
  spark.read
  .format("csv")
  .option("header","True")
  .option("inferSchema","True")
  .csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv")
)
display(df4_usage)

### Header-> False is by default, It is fine if the file has no header, but if the filer has header and if we have given header=False it will consider header as data
### Header-> True if file has header no issues, if file has no header it treats 1st row as column name.
### inferSchema=> False by default, treats all data as string
### inferSchema=> True, samples data and infers a datatype and applies to all rows
#### How schema inference handled “abc” in age? => abc cannot be applied to age so it infers as string

###  5. Column names using string using toDF function for customer data

In [0]:
df_customer=(
    spark.read
    .options(sep=",",inferSchema=True, header=False)
    .csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv")
    .toDF("ID","NAME","AGE","CITY","PLAN TYPE")
)
display(df_customer)

In [0]:
schema1="CUS_ID int, VOICE_MINS int, DATA_GB int,COUNT int"
df_usage=(
    spark.read
    .schema(schema1)
    .options(sep="\t", header=True)
    .csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.tsv")
)
display(df_usage)

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,TimestampType
struct1=StructType([StructField("E_ID", IntegerType(),False),StructField("C_ID", IntegerType(),False),StructField("T_ID", StringType(),False),StructField("SIGNAL_RANGE", IntegerType(),False),StructField("TIME",TimestampType(), False)])
df_tower=(
  spark.read
  .schema(struct1)
  .options(header=True,sep="|")
  .csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_region.csv")
)
display(df_tower)

### 6. Write Operations- CSV format

In [0]:
(
    df_customer
    .write
    .mode("overwrite")
    .csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/clean_data/cus_clean.csv",header=True)
)
display(spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/clean_data/cus_clean.csv",header=True))

In [0]:
(
    df_usage
    .write
    .mode("append")
    .csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/clean_data/usage",header=True)
)
display(spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/clean_data/usage",header=True))

In [0]:
(
    df_tower
    .write
    .options(sep="|", header=True)
    .csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/clean_data/tower_clean.csv")
)

In [0]:
display(spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/clean_data/tower_clean.csv",header=True,sep="|"))

In [0]:
df5=df_tower.show(5)

### 7. Write Operations- JSON format

In [0]:
(
    df_customer
    .write
    .mode("overwrite")
    .json(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/cus_json_clean")
)
display(spark.read.json("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/cus_json_clean"))

In [0]:
(
    df_usage
    .write
    .mode("append")
    .option("compression","snappy")
    .json("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage_json_clean")
)
display(spark.read.json("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage_json_clean"))

In [0]:
(
    df_tower.write
    .mode("ignore")
    .json(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_json_clean")
)
display(spark.read.json(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower_json_clean"))

In [0]:
df_tower1= df_tower.show(5)

### 8. Write Operations- Parquet Format

In [0]:
(
  df_customer
  .write
  .mode("overwrite")
  .option("compression","gzip")
  .parquet("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Parquet/cus_parquet")
)

In [0]:
(
    df_usage
    .write
    .mode("error")
    .parquet(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Parquet/usage_parquet")
)

In [0]:
(
    df_tower
    .write
    .mode("overwrite")
    .option("compression","gzip")
    .parquet(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Parquet/tower_parquet")
)

In [0]:
df_usage1=df_usage.show(5)

### 9. Write Operations- Orc Format

In [0]:
(
  df_customer
  .write
  .mode("overwrite")
  .orc(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Orc/cus_Orc")
)

In [0]:
(
    df_usage
    .write
    .mode("append")
    .orc(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Orc/usage_Orc")
)

In [0]:
(
    df_tower
    .write
    .orc(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Orc/tower_Orc")
)
display(spark.read.orc("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Orc/tower_Orc"))

In [0]:
df_usage2=df_usage.show(5)

### 10. Write Operation- Delta Format

In [0]:
from pyspark.sql.functions import col
df_customer_clean=(
    df_customer.withColumnRenamed("PLAN TYPE","PLAN_TYPE")
)
display(df_customer_clean)

In [0]:
(
  df_customer_clean
  .write
  .mode("overwrite")
  .format("delta")
  .save("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Delta/cus_delta")
)

In [0]:
(
  df_usage
  .write
  .mode("append")
  .format("delta")
  .save(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Delta/usage_delta")
)

In [0]:
(
  df_tower
  .write
  .format("delta")
  .save(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Delta/tower_delta")
)

In [0]:
df_tower_read=(
    spark.read
    .format("delta")
    .load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/Delta/tower_delta")
)
df_tower_read.show(5,False)

### Both Parquet and Delta are parquet format by default, but Delta has logs additionally